For this project I chose to do analysis of the trends of user and critic scores on Rotten Tomatoes. I collected this data myself from webscraping, and I specifically chose to use code from this page: "https://editorial.rottentomatoes.com/guide/oscars-best-and-worst-best-pictures/" because web scraping the whole website was taking too long. Help for this project comes from Stackoverflow, the professor, and my brother.

In [1]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://www.rottentomatoes.com/m/the_super_mario_bros_movie"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

# Movie title
title = soup.title.text.strip()

# Scores
score_data = {
    "Tomatometer": None,
    "Audience Score": None,
    "Critics Avg": None,
    "Audience Avg": None
}

json_tag = soup.find("script", {"id": "media-scorecard-json"})

if json_tag:
    data = json.loads(json_tag.string)

    score_data["Tomatometer"] = data.get("criticsScore", {}).get("score")
    score_data["Audience Score"] = data.get("audienceScore", {}).get("score")
    score_data["Critics Avg"] = data.get("criticsScore", {}).get("averageRating")
    score_data["Audience Avg"] = data.get("audienceScore", {}).get("averageRating")

# Movie info
info = {}

items = soup.find_all("div", {"data-qa": "item"})

for item in items:
    label = item.find("rt-text", {"data-qa": "item-label"})
    values = item.select('[data-qa="item-value"]')  # handles rt-text + rt-link

    if label and values:
        key = label.text.strip()
        val = ", ".join([v.text.strip() for v in values])
        info[key] = val

rating = info.get("Rating")
genre = info.get("Genre")
box_office = info.get("Box Office (Gross USA)")

# Output
print("\n--- Movie Data ---\n")

print(f"{'Title:':25} {title}")
print(f"{'Tomatometer:':25} {score_data['Tomatometer']}")
print(f"{'Audience Score:':25} {score_data['Audience Score']}")
print(f"{'Critics Avg:':25} {score_data['Critics Avg']}")
print(f"{'Audience Avg:':25} {score_data['Audience Avg']}")
print(f"{'Rating:':25} {rating}")
print(f"{'Genre:':25} {genre}")
print(f"{'Box Office (USA):':25} {box_office}")


--- Movie Data ---

Title:                    The Super Mario Bros. Movie | Rotten Tomatoes
Tomatometer:              59
Audience Score:           95
Critics Avg:              5.80
Audience Avg:             4.7
Rating:                   PG (Action and Mild Violence)
Genre:                    Kids & Family, Comedy, Adventure, Animation
Box Office (USA):         $574.9M


The code above was mainly to test if the analysis worked. Once that was done I preceeded below to the rest of my analysis.

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}

GUIDE_URL = "https://editorial.rottentomatoes.com/guide/oscars-best-and-worst-best-pictures/"


# Movie links
def get_movie_links_from_guide():
    res = requests.get(GUIDE_URL, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    movie_urls = set()

    # All Rotten Tomatoes movie pages are /m/...
    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "/m/" in href and "rottentomatoes.com" in href:
            movie_urls.add(href.split("?")[0])

        elif href.startswith("/m/"):
            movie_urls.add("https://www.rottentomatoes.com" + href.split("?")[0])

    return list(movie_urls)


# Scrape 1 movie
def scrape_movie(url):
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    title = soup.title.text.strip() if soup.title else None

    score_data = {
        "Tomatometer": None,
        "Audience Score": None,
        "Critics Avg": None,
        "Audience Avg": None
    }

    json_tag = soup.find("script", {"id": "media-scorecard-json"})
    if json_tag:
        data = json.loads(json_tag.string)
        score_data["Tomatometer"] = data.get("criticsScore", {}).get("score")
        score_data["Audience Score"] = data.get("audienceScore", {}).get("score")
        score_data["Critics Avg"] = data.get("criticsScore", {}).get("averageRating")
        score_data["Audience Avg"] = data.get("audienceScore", {}).get("averageRating")

    info = {}
    for item in soup.find_all("div", {"data-qa": "item"}):
        label = item.find("rt-text", {"data-qa": "item-label"})
        values = item.select('[data-qa="item-value"]')

        if label and values:
            key = label.text.strip()
            val = ", ".join(v.text.strip() for v in values)
            info[key] = val

    return {
        "Title": title,
        "URL": url,
        **score_data,
        "Rating": info.get("Rating"),
        "Genre": info.get("Genre"),
        "Box Office (USA)": info.get("Box Office (Gross USA)")
    }


# Run on list
movie_urls = get_movie_links_from_guide()

print(f"Found {len(movie_urls)} movie links")

results = []

for i, url in enumerate(movie_urls):
    try:
        print(f"[{i+1}/{len(movie_urls)}] Scraping {url}")
        results.append(scrape_movie(url))
        time.sleep(1)  # be polite
    except Exception as e:
        print(f"Failed {url}: {e}")


# Output
df = pd.DataFrame(results)
print(df)

df.to_csv("oscars_best_worst_movies.csv", index=False)

Found 98 movie links
[1/98] Scraping https://www.rottentomatoes.com/m/birdman_2014
[2/98] Scraping https://www.rottentomatoes.com/m/forrest_gump
[3/98] Scraping https://www.rottentomatoes.com/m/terms_of_endearment
[4/98] Scraping https://www.rottentomatoes.com/m/1004177-cimarron
[5/98] Scraping https://www.rottentomatoes.com/m/the_kings_speech
[6/98] Scraping https://www.rottentomatoes.com/m/rain_man
[7/98] Scraping https://www.rottentomatoes.com/m/spotlight_2015
[8/98] Scraping https://www.rottentomatoes.com/m/american_beauty
[9/98] Scraping https://www.rottentomatoes.com/m/american_in_paris
[10/98] Scraping https://www.rottentomatoes.com/m/last_emperor
[11/98] Scraping https://www.rottentomatoes.com/m/mrs_miniver
[12/98] Scraping https://www.rottentomatoes.com/m/moonlight_2016
[13/98] Scraping https://www.rottentomatoes.com/m/chicago
[14/98] Scraping https://www.rottentomatoes.com/m/gigi
[15/98] Scraping https://www.rottentomatoes.com/m/departed
[16/98] Scraping https://www.rottentom

The three questions I had going into this were: Does genre have an influence on whether or not something is an outlier, how do critic scores compare to audience scores, and how does the box office score influence ratings. I used a new type of data visualization that allows you to point to specific parts of the graph and get data off of those points, mostly for user scores which I learned from Stackoverflow. I needed this for researching the aformentioned outliers.

In [ ]:
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_csv("oscars_best_worst_movies.csv")

# Clean data
df["Tomatometer"] = pd.to_numeric(df["Tomatometer"], errors="coerce")
df["Audience Score"] = pd.to_numeric(df["Audience Score"], errors="coerce")
df["Critics Avg"] = pd.to_numeric(df["Critics Avg"], errors="coerce")
df["Audience Avg"] = pd.to_numeric(df["Audience Avg"], errors="coerce")

def parse_box_office(x):
    if pd.isna(x):
        return None
    x = str(x).replace("$", "").replace("M", "")
    try:
        return float(x)
    except:
        return None

df["Box Office (USA)"] = df["Box Office (USA)"].apply(parse_box_office)

df_clean = df.dropna(subset=["Tomatometer", "Audience Score"])

# Fill missing box office
df_clean["Box Office (USA)"] = df_clean["Box Office (USA)"].fillna(0)

# scale for bubble size
df_clean["Box Office Size"] = df_clean["Box Office (USA)"] / 10


# Scatterplot
fig1 = px.scatter(
    df_clean,
    x="Tomatometer",
    y="Audience Score",
    color="Box Office (USA)",
    size="Box Office Size",
    hover_name="Title",
    hover_data=["Critics Avg", "Audience Avg", "Rating", "Genre"],
    title="Critics vs Audience Scores (Bubble = Box Office)"
)
fig1.show()
fig1.write_html("fig1.html")


# Genre Analysis
df_genre = df_clean.assign(Genre=df_clean["Genre"].str.split(", ")).explode("Genre")

fig2 = px.box(
    df_genre,
    x="Genre",
    y="Tomatometer",
    color="Genre",
    title="Critics Scores by Genre"
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()
fig2.write_html("fig2.html")


# Rating Analysis
df_clean["Rating Clean"] = df_clean["Rating"].astype(str).str.split(" ").str[0]

fig3 = px.box(
    df_clean,
    x="Rating Clean",
    y="Audience Score",
    color="Rating Clean",
    title="Audience Scores by MPAA Rating (Cleaned)"
)

fig3.show()
fig3.write_html("fig3.html")


# Critics vs audience
fig4 = px.scatter(
    df_clean,
    x="Critics Avg",
    y="Audience Avg",
    hover_name="Title",
    color="Box Office (USA)",
    size="Box Office Size",
    title="Critics Avg vs Audience Avg Ratings"
)
fig4.show()
fig4.write_html("fig4.html")


# Box Office
fig5 = px.histogram(
    df_clean,
    x="Box Office (USA)",
    nbins=20,
    title="Box Office Distribution (USA)"
)
fig5.show()
fig5.write_html("fig5.html")

Note that box office is in the millions.

Critics vs Audience Scores was the first graph I made, and served as a 'baseline' to analyse the rest of the graphs. Nothing really stood out to me except some small outliers that seemed to all be Romance movies.

Critic Scores By Genre was where I saw most outliers, and I was suprised by the fact that Drama actually had more outliers than Romance did. Though it is not the only one, those two genres seem to be correlated with low score outliers.

Audience Scores by Rating was made when I got curious if there was an easier way to categorize that didn't involve as many categories unlike when sorting by genre. Unfortunately many of the movies I looked at were not rated because they were so old, so it was less helpful then I wanted.

Critic Avg vs Audience Avg was made to get a better look at critics vs audience without using the percentage system. Audiences tend to rate movies higher than critics do on average, but critics have a more even distribution of scores across the graph.

Box Office Distribution is the simplest graph but I wanted a comparison across all movies. I was suprised by how little most movies made, I was expecting it to be more evenly distributed instead of skewed so far. Box office score doesn't really seem to have any correlation to scores, which suprised me.

The hardest part of this project was definetely wrangling the code. The MPAA scores has many trigger warnings attached to them and I had a very difficult time cleaning up the data.